# Détection de Pneumonie sur Radiographies Thoraciques
## Classification Binaire : NORMAL vs PNEUMONIA


## Configuration & Vérification GPU

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow opencv-python kaggle

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import zipfile
import random
import time
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import plot_model

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_recall_curve, average_precision_score
)
import cv2

In [ ]:
main_path = "/content"
os.makedirs(os.path.join(main_path, 'tmp'), exist_ok=True)

In [ ]:
# ── Seed de reproductibilité ──────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

In [ ]:
# ── Vérification GPU ──────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU détecté : {gpus[0].name}')
    # Afficher la mémoire GPU disponible
    try:
        gpu_info = !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader,nounits 2>/dev/null
        print(f'   {gpu_info[0] if gpu_info else "Info GPU non disponible"}')
    except:
        pass
else:
    print('Aucun GPU détecté — exécution sur CPU (plus lent)')

In [ ]:
# ── Constantes globales ───────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_P1   = 15
EPOCHS_P2   = 20
LR_P1       = 1e-3
LR_P2       = 1e-5
CLASSES     = ['NORMAL', 'PNEUMONIA']
MODEL_NAME  = 'DenseNet121_Pneumonia'
SAVE_DIR    = f'{main_path}/Models/'
DPI         = 300 # Global DPI for saving figures

print(f'Hyperparamètres :')
print(f'   IMG_SIZE   : {IMG_SIZE}x{IMG_SIZE}')
print(f'   BATCH_SIZE : {BATCH_SIZE}')
print(f'   EPOCHS P1  : {EPOCHS_P1} | LR : {LR_P1}')
print(f'   EPOCHS P2  : {EPOCHS_P2} | LR : {LR_P2}')

In [ ]:
import sys
!{sys.executable} -m pip install kaggle

# Configuration Kaggle & Téléchargement

In [ ]:
from google.colab import userdata, files
import os
import zipfile
import time
import json

In [ ]:
# ── Configuration Kaggle API ──────────────────────────────────
KAGGLE_CONFIG_DIR = '/root/.kaggle/'
os.makedirs(KAGGLE_CONFIG_DIR, exist_ok=True)

In [ ]:
uploaded = files.upload()  # Prompts user to select a file

for filename, content_bytes in uploaded.items():
    content_str = content_bytes.decode('utf-8')
    kaggle_creds = json.loads(content_str)
    os.environ['KAGGLE_USERNAME'] = kaggle_creds['username']
    os.environ['KAGGLE_KEY'] = kaggle_creds['key']
    print('Kaggle credentials set from uploaded file.')

In [ ]:
# ── Chemins ───────────────────────────────────────────────────
KAGGLE_DATASET_PATH = 'mahabubalam31/chest-x-ray-dataset-for-pneumoniabalanced'
DATA_DIR = f'{main_path}/chest_xray'
ZIP_FILE_NAME = 'chest-x-ray-dataset-for-pneumoniabalanced.zip'
ZIP_PATH = os.path.join(main_path, ZIP_FILE_NAME)

In [ ]:
# ── Téléchargement Kaggle ─────────────────────────────────────
if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f'⚡ Données déjà décompressées dans {DATA_DIR}. Saut du téléchargement.')
elif os.path.exists(ZIP_PATH):
    print(f'📦 Fichier zip déjà téléchargé : {ZIP_PATH}. Saut du téléchargement.')
else:
    print(f'⬇️ Téléchargement du dataset Kaggle : {KAGGLE_DATASET_PATH}...')
    t0 = time.time()
    !kaggle datasets download -d {KAGGLE_DATASET_PATH} -p {main_path}
    print(f'✅ Téléchargement terminé en {time.time()-t0:.1f}s')

In [ ]:
import os
import time
import zipfile

# ── Décompression ─────────────────────────────────────────────
# Re-evaluate ZIP_PATH to reflect the actual download location due to Kaggle's -p behavior
ZIP_PATH = os.path.join(main_path, ZIP_FILE_NAME)

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f'⚡ Données déjà décompressées dans {DATA_DIR}.')
else:
    print(f'📂 Décompression vers {DATA_DIR}...')
    os.makedirs(DATA_DIR, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        print(f'   Nombre de fichiers dans l\'archive : {len(zf.namelist())}')
        zf.extractall(DATA_DIR)
    print(f'✅ Décompression terminée en {time.time()-t0:.1f}s')

In [ ]:
# ── Détection automatique de la structure ────────────────────
def find_split_dirs(base):
    """Trouve automatiquement les dossiers TRAIN/VAL/TEST"""
    splits = {}
    for root, dirs, files in os.walk(base):
        dirname = os.path.basename(root).upper()
        if dirname in ['TRAIN', 'TRAINING']:
            splits['train'] = root
        elif dirname in ['VAL', 'VALID', 'VALIDATION']:
            splits['val'] = root
        elif dirname in ['TEST', 'TESTING']:
            splits['test'] = root
    return splits

splits = find_split_dirs(DATA_DIR)

In [ ]:
# Fallback si structure différente
if not splits:
    # Chercher directement sous chest_xray/
    for name in os.listdir(DATA_DIR):
        full = os.path.join(DATA_DIR, name)
        if os.path.isdir(full):
            key = name.lower()
            if 'train' in key: splits['train'] = full
            elif 'val'   in key: splits['val']   = full
            elif 'test'  in key: splits['test']  = full

In [ ]:
# Specific adjustment for this dataset structure if needed
# The dataset typically extracts to 'chest_xray/chest_xray_balanced/train', etc.
if not splits.get('train') and os.path.exists(os.path.join(DATA_DIR, 'chest_xray_balanced')):
    base_dataset_dir = os.path.join(DATA_DIR, 'chest_xray_balanced')
    for name in os.listdir(base_dataset_dir):
        full = os.path.join(base_dataset_dir, name)
        if os.path.isdir(full):
            key = name.lower()
            if 'train' in key: splits['train'] = full
            elif 'val'   in key: splits['val']   = full
            elif 'test'  in key: splits['test']  = full

TRAIN_DIR = splits.get('train')
VAL_DIR   = splits.get('val')
TEST_DIR  = splits.get('test')

print('Structure des données détectée :')
for split, path in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('TEST', TEST_DIR)]:
    if path:
        print(f'   {split:6s} → {path}')
    else:
        print(f'   {split:6s} → ⚠️  Non trouvé !')

# Vérification des sous-dossiers de classes
print('Vérification des classes :')
for split, path in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('TEST', TEST_DIR)]:
    if path and os.path.exists(path):
        subdirs = sorted(os.listdir(path))
        print(f'   {split:6s} : {subdirs}')
    else:
        print(f'   {split:6s} : Dossier non trouvé ou vide.')

print('Structure validée')

## 📊 Cellule 3 — EDA : Analyse Exploratoire des Données

In [ ]:
def compter_images(split_dir):
    """Compte les images par classe dans un dossier split"""
    stats = {}
    if not split_dir or not os.path.exists(split_dir):
        return stats
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            imgs = [f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            stats[cls] = len(imgs)
    return stats


In [ ]:
# ── Statistiques générales ────────────────────────────────────
stats = {
    'Train' : compter_images(TRAIN_DIR),
    'Val'   : compter_images(VAL_DIR),
    'Test'  : compter_images(TEST_DIR),
}

total_global = 0
for split, counts in stats.items():
    total = sum(counts.values())
    total_global += total
    print(f'\n  {split.upper()} ({total} images) :')
    for cls, n in counts.items():
        pct = n / total * 100 if total > 0 else 0
        bar = '█' * int(pct / 2)
        print(f'    {cls:12s} : {n:5d} ({pct:5.1f}%)  {bar}')
print(f'TOTAL GLOBAL : {total_global} images')

**Distribution par split**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribution des Classes par Split', fontsize=16, fontweight='bold', y=1.02)

colors = {'NORMAL': '#2ecc71', 'PNEUMONIA': '#e74c3c'}

for ax, (split, counts) in zip(axes, stats.items()):
    if not counts:
        ax.text(0.5, 0.5, 'Données\nnon disponibles',
                ha='center', va='center', transform=ax.transAxes)
        continue
    classes = list(counts.keys())
    values  = list(counts.values())
    bar_colors = [colors.get(c, '#3498db') for c in classes]

    bars = ax.bar(classes, values, color=bar_colors, edgecolor='white',
                  linewidth=1.5, width=0.6)
    ax.set_title(f'{split}\n({sum(values)} images)', fontweight='bold', fontsize=13)
    ax.set_ylabel('Nombre d\'images')
    ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(f'{main_path}/tmp/eda_distribution.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2 : Exemples d'images par classe ───────────────────

def charger_exemples(split_dir, n=4):
    """Charge n exemples aléatoires par classe"""
    exemples = {}
    if not split_dir or not os.path.exists(split_dir):
        return exemples
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        chosen = random.sample(imgs, min(n, len(imgs)))
        exemples[cls] = [os.path.join(cls_path, f) for f in chosen]
    return exemples

exemples = charger_exemples(TRAIN_DIR, n=4)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Exemples de Radiographies Thoraciques — Dataset TRAIN',
             fontsize=16, fontweight='bold', y=1.01)

cls_titles = {'NORMAL': ('Classe : NORMAL ✅', '#2ecc71'),
              'PNEUMONIA': ('Classe : PNEUMONIA 🔴', '#e74c3c')}

for row, (cls, paths) in enumerate(exemples.items()):
    title_text, title_color = cls_titles.get(cls, (cls, 'gray'))
    for col, img_path in enumerate(paths[:4]):
        ax = axes[row][col]
        img = plt.imread(img_path)
        ax.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(title_color)
            spine.set_linewidth(3)
        if col == 0:
            ax.set_ylabel(title_text, color=title_color,
                          fontweight='bold', fontsize=11)
        fname = os.path.basename(img_path)
        # Afficher dimensions de l'image
        h, w = img.shape[:2]
        ax.set_title(f'{fname[:20]}\n{w}×{h}px', fontsize=8)

plt.tight_layout()
plt.savefig(f'{main_path}/tmp/eda_exemples.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('✅ Figure 2 sauvegardée')

**Analyse des dimensions et statistiques**

In [ ]:
widths, heights, channels_list = [], [], []
sample_size = min(200, sum(compter_images(TRAIN_DIR).values()))

all_imgs = []
for cls in os.listdir(TRAIN_DIR):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        imgs = [os.path.join(cls_path, f)
                for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        all_imgs.extend(imgs)

sample_imgs = random.sample(all_imgs, min(sample_size, len(all_imgs)))

for path in sample_imgs:
    try:
        img = plt.imread(path)
        if len(img.shape) == 2:
            h, w = img.shape
            c = 1
        else:
            h, w, c = img.shape
        widths.append(w)
        heights.append(h)
        channels_list.append(c)
    except:
        pass

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Analyse des Propriétés des Images (n={len(widths)})',
             fontsize=14, fontweight='bold')

# Largeurs
axes[0].hist(widths, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(np.mean(widths), color='red', linestyle='--',
                label=f'Moy: {np.mean(widths):.0f}px')
axes[0].set_title('Distribution des Largeurs')
axes[0].set_xlabel('Largeur (px)')
axes[0].set_ylabel('Fréquence')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Hauteurs
axes[1].hist(heights, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1].axvline(np.mean(heights), color='red', linestyle='--',
                label=f'Moy: {np.mean(heights):.0f}px')
axes[1].set_title('Distribution des Hauteurs')
axes[1].set_xlabel('Hauteur (px)')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

# Canaux
channel_counts = pd.Series(channels_list).value_counts()
axes[2].bar(channel_counts.index.astype(str), channel_counts.values,
            color=['#e74c3c', '#2ecc71', '#3498db'][:len(channel_counts)],
            edgecolor='white')
axes[2].set_title('Canaux de Couleur')
axes[2].set_xlabel('Nombre de canaux (1=Gris, 3=RGB)')
axes[2].set_ylabel('Fréquence')
axes[2].spines[['top','right']].set_visible(False)
for i, (idx, val) in enumerate(channel_counts.items()):
    axes[2].text(i, val + 1, str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{main_path}/tmp/eda_dimensions.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
print(f'Statistiques des dimensions :')
print(f'   Largeur  : min={min(widths)}  max={max(widths)}  moy={np.mean(widths):.0f}  med={np.median(widths):.0f}')
print(f'   Hauteur  : min={min(heights)} max={max(heights)} moy={np.mean(heights):.0f} med={np.median(heights):.0f}')
print(f'   Canaux   : {dict(pd.Series(channels_list).value_counts())}')

## Prétraitement

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255)

In [ ]:
# Validation & Test : uniquement normalisation
val_test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# ── Générateurs ───────────────────────────────────────────────
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size   = (IMG_SIZE, IMG_SIZE),
    batch_size    = BATCH_SIZE,
    class_mode    = 'binary',
    shuffle       = True,
    seed          = SEED,
    color_mode    = 'rgb'
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    shuffle     = False,
    color_mode  = 'rgb'
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    shuffle     = False,
    color_mode  = 'rgb'
)

In [ ]:
print('Résumé des générateurs :')
print(f'Train   : {train_gen.n} images | {len(train_gen)} batches')
print(f'Val     : {val_gen.n}   images | {len(val_gen)} batches')
print(f'Test    : {test_gen.n}  images | {len(test_gen)} batches')
print(f'Correspondance classes : {train_gen.class_indices}')

In [ ]:
# Sauvegarder le mapping des classes
CLASS_INDICES = train_gen.class_indices
IDX_TO_CLASS  = {v: k for k, v in CLASS_INDICES.items()}

## Architecture DenseNet121

In [ ]:
def construire_modele(img_size=224, unfreeze_layers=0, dropout1=0.4, dropout2=0.3):
    base_model = DenseNet121(weights='imagenet', include_top=False,
                             input_shape=(img_size, img_size, 3))
    for layer in base_model.layers:
        layer.trainable = False
    if unfreeze_layers > 0:
        for layer in base_model.layers[-unfreeze_layers:]:
            if not isinstance(layer, layers.BatchNormalization):
                layer.trainable = True

    x = base_model.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, name='fc_256')(x)
    x = layers.BatchNormalization(name='bn_256')(x)
    x = layers.Activation('relu', name='act_256')(x)
    x = layers.Dropout(dropout1, name='drop_256')(x)   # <-- use parameter
    x = layers.Dense(64, name='fc_64')(x)
    x = layers.BatchNormalization(name='bn_64')(x)
    x = layers.Activation('relu', name='act_64')(x)
    x = layers.Dropout(dropout2, name='drop_64')(x)   # <-- use parameter
    output = layers.Dense(1, activation='sigmoid', name='predictions')(x)

    model = Model(inputs=base_model.input, outputs=output,
                  name=f'{MODEL_NAME}_unfreeze{unfreeze_layers}')
    return model, base_model

In [ ]:
model, base_model = construire_modele(IMG_SIZE, unfreeze_layers=0)

In [ ]:
def get_shape(layer):
    try:
        return str(layer.output_shape)
    except AttributeError:
        return "(forme non disponible)"

In [ ]:
# Résumé
trainable     = sum(p.numpy().size for p in model.trainable_weights)
non_trainable = sum(p.numpy().size for p in model.non_trainable_weights)
total         = trainable + non_trainable

print(f'Paramètres du modèle :')
print(f'   Entraînables     : {trainable:>12,.0f}')
print(f'   Non-entraînables : {non_trainable:>12,.0f}')
print(f'   TOTAL            : {total:>12,.0f}')
print(f'   Couches backbone : {len(base_model.layers)}')
print(f'   Couches totales  : {len(model.layers)}')

In [ ]:
# Afficher les dernières couches du backbone
print(f'Dernières couches backbone :')
for layer in base_model.layers[-5:]:
    status = "✓" if layer.trainable else "✗"
    print(f'   [{status}] {layer.name:40s} {get_shape(layer)}')

print(f'Couches de la tête :')
for layer in model.layers[-8:]:
    status = "✓" if layer.trainable else "✗"
    print(f'   [{status}] {layer.name:40s} {get_shape(layer)}')

print('Modèle construit avec succès')

### Hyperparameter Tuning: Grid Search

In [ ]:
def run_training_experiment(lr_p1, dropout1, dropout2, unfreeze_layers_p2,
                            train_generator, val_generator, test_generator,
                            epochs_p1=EPOCHS_P1, epochs_p2=EPOCHS_P2):
    print(f"\n--- Expérience: LR_P1={lr_p1}, Dropouts=({dropout1}, {dropout2}), Unfreeze_P2={unfreeze_layers_p2} ---")

    # Phase 1: Transfer Learning
    # The construire_modele function now accepts dropout values
    model, base_model = construire_modele(IMG_SIZE, unfreeze_layers=0,
                                         dropout1=dropout1, dropout2=dropout2)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr_p1),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )

    cb_early = callbacks.EarlyStopping(
        monitor='val_auc', patience=5, restore_best_weights=True, mode='max', verbose=0
    )
    cb_reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0
    )

    history_p1 = model.fit(
        train_generator,
        epochs=epochs_p1,
        validation_data=val_generator,
        callbacks=[cb_early, cb_reduce_lr],
        verbose=0
    )
    # Get best val_auc from Phase 1 history
    val_auc_p1 = max(history_p1.history['val_auc']) if 'val_auc' in history_p1.history and history_p1.history['val_auc'] else 0
    print(f"  Phase 1 terminée (epochs: {len(history_p1.history['loss'])}) - Best val_auc: {val_auc_p1:.4f}")

    # Phase 2: Fine-Tuning
    # Unfreeze layers based on unfreeze_layers_p2
    for layer in base_model.layers:
        layer.trainable = False
    if unfreeze_layers_p2 > 0:
        for layer in base_model.layers[-unfreeze_layers_p2:]:
            if not isinstance(layer, layers.BatchNormalization):
                layer.trainable = True

    # Use a lower learning rate for fine-tuning, relative to lr_p1
    lr_p2_tuned = lr_p1 / 100 # Example: 1e-5 if lr_p1 is 1e-3
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr_p2_tuned),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )

    cb_early_p2 = callbacks.EarlyStopping(
        monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=0
    )
    cb_reduce_lr_p2 = callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=0
    )

    history_p2 = model.fit(
        train_generator,
        epochs=epochs_p2,
        validation_data=val_generator,
        callbacks=[cb_early_p2, cb_reduce_lr_p2],
        verbose=0
    )
    # Get best val_auc from Phase 2 history
    val_auc_p2 = max(history_p2.history['val_auc']) if 'val_auc' in history_p2.history and history_p2.history['val_auc'] else 0
    print(f"  Phase 2 terminée (epochs: {len(history_p2.history['loss'])}) - Best val_auc: {val_auc_p2:.4f}")

    # Final evaluation on the test set using the best model from Phase 2
    test_generator.reset() # Reset generator for consistent evaluation
    y_prob = model.predict(test_generator, verbose=0).flatten()
    y_true = test_generator.classes

    # Find optimal threshold using F1-score
    thresholds = np.arange(0.1, 0.9, 0.01)
    f1_scores = [f1_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
    optimal_thresh = thresholds[np.argmax(f1_scores)]

    y_pred_opt = (y_prob >= optimal_thresh).astype(int)

    accuracy = (y_pred_opt == y_true).mean()
    auc_roc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred_opt)

    print(f"  Résultats finaux (Test): Accuracy={accuracy:.4f}, AUC={auc_roc:.4f}, F1={f1:.4f}")

    return {'lr_p1': lr_p1, 'dropout1': dropout1, 'dropout2': dropout2,
            'unfreeze_layers_p2': unfreeze_layers_p2,
            'accuracy': accuracy, 'auc': auc_roc, 'f1': f1, 'optimal_threshold': optimal_thresh}

In [ ]:
# Parameter grid (expanded to include unfreeze_layers_p2)
param_grid = [
    {'lr_p1': 1e-3, 'dropout1': 0.3, 'dropout2': 0.3, 'unfreeze_layers_p2': 0},
    {'lr_p1': 1e-3, 'dropout1': 0.3, 'dropout2': 0.3, 'unfreeze_layers_p2': 50},
    {'lr_p1': 1e-3, 'dropout1': 0.5, 'dropout2': 0.5, 'unfreeze_layers_p2': 0},
    {'lr_p1': 5e-4, 'dropout1': 0.3, 'dropout2': 0.3, 'unfreeze_layers_p2': 50},
    {'lr_p1': 5e-4, 'dropout1': 0.5, 'dropout2': 0.5, 'unfreeze_layers_p2': 100}
]

results = []
best_test_auc = -1
best_params = None

print("--- Démarrage de la Grid Search --- ")
print(f"   Exploration de {len(param_grid)} combinaisons.")

for i, params in enumerate(param_grid):
    print(f"\n--- Exécution de l'expérience {i+1}/{len(param_grid)} ---")

    exp_results = run_training_experiment(
        lr_p1=params['lr_p1'],
        dropout1=params['dropout1'],
        dropout2=params['dropout2'],
        unfreeze_layers_p2=params['unfreeze_layers_p2'],
        train_generator=train_gen,
        val_generator=val_gen,
        test_generator=test_gen,
        epochs_p1=EPOCHS_P1,
        epochs_p2=EPOCHS_P2
    )
    results.append(exp_results)

    if exp_results['auc'] > best_test_auc: # Using AUC from test set for best params
        best_test_auc = exp_results['auc']
        best_params = params

print("\n--- Grid Search terminée ---")

print(f"\n✅ Meilleurs hyperparamètres trouvés (basé sur AUC du TEST set):")
print(f"   LR_P1               : {best_params['lr_p1']}")
print(f"   Dropout 1           : {best_params['dropout1']}")
print(f"   Dropout 2           : {best_params['dropout2']}")
print(f"   Couches débloquées P2: {best_params['unfreeze_layers_p2']}")
print(f"   Meilleur AUC (Test) : {best_test_auc:.4f}")

# Update global constants with the best parameters for the subsequent full training
# (Note: EPOCHS_P1/P2 are not tuned here, they are max values for EarlyStopping)
LR_P1 = best_params['lr_p1']
# LR_P2 is calculated dynamically inside run_training_experiment as lr_p1 / 100, so update LR_P2 global constant for consistency with later compilation
LR_P2 = LR_P1 / 100
UNFREEZE_FROM = best_params['unfreeze_layers_p2']

# Display results dataframe
results_df = pd.DataFrame(results)
display(results_df.sort_values(by='auc', ascending=False))


--- Démarrage de la Grid Search --- 
   Exploration de 5 combinaisons.

--- Exécution de l'expérience 1/5 ---

--- Expérience: LR_P1=0.001, Dropouts=(0.3, 0.3), Unfreeze_P2=0 ---


In [ ]:
# Rebuild the final model with best hyperparameters for Phase 1 (and dropout values)
# Note: dropout values are now from best_params, and LR_P1 is updated globally.
# UNFREEZE_FROM will be used later for Phase 2 setup.
model, base_model = construire_modele(IMG_SIZE,
                                      unfreeze_layers=0, # Initial model for Phase 1 starts frozen
                                      dropout1=best_params['dropout1'],
                                      dropout2=best_params['dropout2'])

## Phase 1 : Transfer Learning (têtes uniquement)

In [ ]:
# ── Compilation ───────────────────────────────────────────────
model.compile(
    optimizer = optimizers.Adam(learning_rate=LR_P1),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [ ]:
# ── Callbacks Phase 1 ─────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)

cb_early = callbacks.EarlyStopping(
    monitor              = 'val_auc',
    patience             = 5,
    restore_best_weights = True,
    mode                 = 'max',
    verbose              = 1
)

cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 3,
    min_lr   = 1e-7,
    verbose  = 1
)

cb_checkpoint_p1 = callbacks.ModelCheckpoint(
    filepath          = os.path.join(SAVE_DIR, f'{MODEL_NAME}_phase1_best.keras'),
    monitor           = 'val_auc',
    save_best_only    = True,
    mode              = 'max',
    verbose           = 1
)

cb_csv_p1 = callbacks.CSVLogger(
    os.path.join(SAVE_DIR, f'{MODEL_NAME}_phase1_log.csv'),
    append=False
)

In [ ]:
# ── Entraînement Phase 1 ──────────────────────────────────────
t0 = time.time()
history_p1 = model.fit(
    train_gen,
    epochs          = EPOCHS_P1,
    validation_data = val_gen,
    callbacks       = [cb_early, cb_reduce_lr, cb_checkpoint_p1, cb_csv_p1],
    verbose         = 1
)

In [ ]:
duree_p1 = time.time() - t0
print(f'Durée Phase 1 : {duree_p1/60:.1f} minutes')
print(f'   Meilleur val_auc : {max(history_p1.history["val_auc"]):.4f}')
print(f'   Meilleure val_acc: {max(history_p1.history["val_accuracy"]):.4f}')

In [ ]:
def tracer_courbes(history, titre='', phase='1'):
    """Trace les courbes d'entraînement"""
    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'Courbes d\'Entraînement — {titre}',
                 fontsize=15, fontweight='bold')

    metrics = [
        ('loss',      'val_loss',      'Loss (Binary Crossentropy)', '#e74c3c'),
        ('accuracy',  'val_accuracy',  'Accuracy',                   '#2ecc71'),
        ('auc',       'val_auc',       'AUC-ROC',                    '#3498db'),
        ('precision', 'val_precision', 'Précision',                  '#9b59b6'),
    ]

    for ax, (train_key, val_key, title, color) in zip(axes.flat, metrics):
        ax.plot(epochs, h[train_key], color=color, linewidth=2,
                marker='o', markersize=4, label='Train')
        ax.plot(epochs, h[val_key], color=color, linewidth=2,
                linestyle='--', marker='s', markersize=4, label='Val',
                alpha=0.8)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Époque')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.spines[['top','right']].set_visible(False)

        # Marquer le meilleur point
        if 'loss' in val_key:
            best_ep = np.argmin(h[val_key]) + 1
            best_val = min(h[val_key])
        else:
            best_ep = np.argmax(h[val_key]) + 1
            best_val = max(h[val_key])
        ax.axvline(x=best_ep, color='gray', linestyle=':',
                   alpha=0.7, label=f'Best: ép.{best_ep}')
        ax.text(best_ep, best_val, f' {best_val:.3f}',
                color='black', fontsize=8, va='bottom')
        ax.legend(fontsize=9)

    plt.tight_layout()
    path = f'{main_path}/tmp/courbes_phase{phase}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    return path

In [ ]:
def tracer_courbes(history, titre='', phase='1', dpi=DPI):
    """Trace les courbes d'entraînement"""
    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'Courbes d\'Entraînement — {titre}',
                 fontsize=15, fontweight='bold')

    metrics = [
        ('loss',      'val_loss',      'Loss (Binary Crossentropy)', '#e74c3c'),
        ('accuracy',  'val_accuracy',  'Accuracy',                   '#2ecc71'),
        ('auc',       'val_auc',       'AUC-ROC',                    '#3498db'),
        ('precision', 'val_precision', 'Précision',                  '#9b59b6'),
    ]

    for ax, (train_key, val_key, title, color) in zip(axes.flat, metrics):
        ax.plot(epochs, h[train_key], color=color, linewidth=2,
                marker='o', markersize=4, label='Train')
        ax.plot(epochs, h[val_key], color=color, linewidth=2,
                linestyle='--', marker='s', markersize=4, label='Val',
                alpha=0.8)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Époque')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.spines[['top','right']].set_visible(False)

        # Marquer le meilleur point
        if 'loss' in val_key:
            best_ep = np.argmin(h[val_key]) + 1
            best_val = min(h[val_key])
        else:
            best_ep = np.argmax(h[val_key]) + 1
            best_val = max(h[val_key])
        ax.axvline(x=best_ep, color='gray', linestyle=':',
                   alpha=0.7, label=f'Best: ép.{best_ep}')
        ax.text(best_ep, best_val, f' {best_val:.3f}',
                color='black', fontsize=8, va='bottom')
        ax.legend(fontsize=9)

    plt.tight_layout()
    path = f'{main_path}/tmp/courbes_phase{phase}.png'
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.show()
    return path

In [ ]:
tracer_courbes(history_p1, titre='Phase 1 — Transfer Learning', phase='1', dpi=DPI)

## Phase 2 : Fine-Tuning

**Débloquer les dernières couches du backbone**

In [ ]:
UNFREEZE_FROM = 50

for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-UNFREEZE_FROM:]:
    # Ne pas débloquer les BatchNorm (évite l'instabilité)
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

# Compter les couches débloquées
n_trainable_after = sum(1 for l in model.layers if l.trainable)
trainable_p2 = sum(p.numpy().size for p in model.trainable_weights)

print(f'Couches entraînables : {n_trainable_after}')
print(f'Paramètres entraîn. : {trainable_p2:,.0f}')

In [ ]:
# ── Recompiler avec LR plus faible ────────────────────────────
model.compile(
    optimizer = optimizers.Adam(learning_rate=LR_P2),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [ ]:
# ── Callbacks Phase 2 ─────────────────────────────────────────
cb_early_p2 = callbacks.EarlyStopping(
    monitor              = 'val_auc',
    patience             = 7,
    restore_best_weights = True,
    mode                 = 'max',
    verbose              = 1
)

cb_reduce_lr_p2 = callbacks.ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 3,
    min_lr   = 1e-8,
    verbose  = 1
)

cb_checkpoint_final = callbacks.ModelCheckpoint(
    filepath       = os.path.join(SAVE_DIR, f'{MODEL_NAME}_FINAL.keras'),
    monitor        = 'val_auc',
    save_best_only = True,
    mode           = 'max',
    verbose        = 1
)

cb_csv_p2 = callbacks.CSVLogger(
    os.path.join(SAVE_DIR, f'{MODEL_NAME}_phase2_log.csv'),
    append=False
)

In [ ]:
# ── Entraînement Phase 2 ──────────────────────────────────────
t0 = time.time()
history_p2 = model.fit(
    train_gen,
    epochs          = EPOCHS_P2,
    validation_data = val_gen,
    callbacks       = [cb_early_p2, cb_reduce_lr_p2, cb_checkpoint_final, cb_csv_p2],
    verbose         = 1
)

In [ ]:
duree_p2 = time.time() - t0
print(f'Durée Phase 2 : {duree_p2/60:.1f} minutes')
print(f'   Meilleur val_auc : {max(history_p2.history["val_auc"]):.4f}')
print(f'   Meilleure val_acc: {max(history_p2.history["val_accuracy"]):.4f}')
print(f'Durée totale : {(duree_p1+duree_p2)/60:.1f} minutes')

In [ ]:
# ── Courbes Phase 2 ───────────────────────────────────────────
tracer_courbes(history_p2, titre='Phase 2 — Fine-Tuning', phase='2', dpi=DPI)

## Évaluation Complète sur TEST

In [ ]:
# Charger le meilleur modèle sauvegardé
best_model_path = os.path.join(SAVE_DIR, f'{MODEL_NAME}_FINAL.keras')
if os.path.exists(best_model_path):
    print(f'   Chargement du meilleur modèle : {best_model_path}')
    model = keras.models.load_model(best_model_path)
    print('   ✅ Modèle chargé')
else:
    print('   ⚠️  Meilleur modèle non trouvé, utilisation du modèle courant')

In [ ]:
# ── Prédictions sur TEST ──────────────────────────────────────
test_gen.reset()  # Important : remettre à zéro
y_prob  = model.predict(test_gen, verbose=1) # proba [0,1]
y_prob  = y_prob.flatten()
y_true  = test_gen.classes

print(f'Prédictions générées : {len(y_prob)}')
print(f'Labels vrais         : {len(y_true)}')
print(f'Distribution labels  : NORMAL={sum(y_true==0)}, PNEUMONIA={sum(y_true==1)}')

In [ ]:
# ── Métriques globales avec seuil 0.5 ─────────────────────────
test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(test_gen, verbose=0)

print(f'MÉTRIQUES TEST (seuil=0.5) :')
print(f'   Loss      : {test_loss:.4f}')
print(f'   Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'   AUC-ROC   : {test_auc:.4f}')
print(f'   Précision : {test_prec:.4f}')
print(f'   Rappel    : {test_rec:.4f}')

In [ ]:
# ── Calcul du seuil optimal via F1-score ──────────────────────
print('Recherche du seuil optimal (F1-score)...')
thresholds   = np.arange(0.1, 0.9, 0.01)
f1_scores    = [f1_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
OPTIMAL_THRESH = thresholds[np.argmax(f1_scores)]
OPTIMAL_F1     = max(f1_scores)

print(f'Seuil optimal : {OPTIMAL_THRESH:.2f}')
print(f'F1 optimal    : {OPTIMAL_F1:.4f}')

In [ ]:
# Prédictions avec seuil optimal
y_pred_opt = (y_prob >= OPTIMAL_THRESH).astype(int)
y_pred_def = (y_prob >= 0.5).astype(int)

In [ ]:
# ── Rapport de classification ─────────────────────────────────
print(f'RAPPORT DE CLASSIFICATION (seuil={OPTIMAL_THRESH:.2f}) :')
print(classification_report(
    y_true, y_pred_opt,
    target_names=['NORMAL', 'PNEUMONIA'],
    digits=4
))

In [ ]:
# ── Courbe F1 vs Seuil ────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(thresholds, f1_scores, 'b-', linewidth=2)
ax.axvline(OPTIMAL_THRESH, color='red', linestyle='--',
           label=f'Seuil optimal={OPTIMAL_THRESH:.2f}\nF1={OPTIMAL_F1:.4f}')
ax.axvline(0.5, color='gray', linestyle=':', label='Seuil par défaut=0.5')
ax.set_title('F1-Score en fonction du Seuil de Décision', fontweight='bold')
ax.set_xlabel('Seuil')
ax.set_ylabel('F1-Score')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{main_path}/tmp/f1_seuil.png', dpi=DPI)
plt.show()

In [ ]:
# ── Matrice de Confusion ──────────────────────────────────────

def tracer_matrice_confusion(y_true, y_pred, title='Matrice de Confusion',
                              classes=['NORMAL', 'PNEUMONIA'], dpi=DPI):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=15, fontweight='bold')

    # Matrice absolue
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=axes[0], linewidths=0.5,
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[0].set_title('Valeurs Absolues', fontweight='bold')
    axes[0].set_ylabel('Vrai Label')
    axes[0].set_xlabel('Label Prédit')

    # Matrice normalisée
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
                xticklabels=classes, yticklabels=classes,
                ax=axes[1], linewidths=0.5,
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[1].set_title('Valeurs Normalisées (%)', fontweight='bold')
    axes[1].set_ylabel('Vrai Label')
    axes[1].set_xlabel('Label Prédit')

    # Métriques
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    ppv = tp / (tp + fp)
    npv = tn / (tn + fn)

    info = (f'TP={tp}  TN={tn}  FP={fp}  FN={fn}\n'
            f'Sensibilité(Rappel)={sensitivity:.3f}  Spécificité={specificity:.3f}\n'
            f'VPP={ppv:.3f}  VPN={npv:.3f}')
    fig.text(0.5, -0.02, info, ha='center', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.savefig(f'{main_path}/tmp/matrice_confusion.png', dpi=dpi, bbox_inches='tight')
    plt.show()
    return cm

cm = tracer_matrice_confusion(y_true, y_pred_opt,
    title=f'Matrice de Confusion — TEST (seuil={OPTIMAL_THRESH:.2f})', dpi=DPI)

In [ ]:
# ── Courbes ROC et Précision-Rappel ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Courbes de Performance', fontsize=15, fontweight='bold')

# ─ Courbe ROC ─
fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)

axes[0].plot(fpr, tpr, 'b-', linewidth=2,
             label=f'DenseNet121 (AUC = {roc_auc:.4f})')
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5, label='Classif. aléatoire')

# Marquer le point optimal
idx_opt = np.argmin(np.abs(roc_thresholds - OPTIMAL_THRESH))
axes[0].scatter(fpr[idx_opt], tpr[idx_opt], color='red', s=150, zorder=5,
                label=f'Seuil optimal ({OPTIMAL_THRESH:.2f})')

axes[0].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0].set_title('Courbe ROC', fontweight='bold')
axes[0].set_xlabel('Taux de Faux Positifs')
axes[0].set_ylabel('Taux de Vrais Positifs (Sensibilité)')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# ─ Courbe Précision-Rappel ─
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)

axes[1].plot(recall_vals, precision_vals, 'g-', linewidth=2,
             label=f'DenseNet121 (AP = {ap:.4f})')
baseline = sum(y_true) / len(y_true)
axes[1].axhline(baseline, color='k', linestyle='--', alpha=0.5,
                label=f'Baseline ({baseline:.3f})')

# Point optimal
if len(pr_thresholds) > 0:
    idx_opt_pr = np.argmin(np.abs(pr_thresholds - OPTIMAL_THRESH))
    axes[1].scatter(recall_vals[idx_opt_pr], precision_vals[idx_opt_pr],
                    color='red', s=150, zorder=5,
                    label=f'Seuil optimal ({OPTIMAL_THRESH:.2f})')

axes[1].fill_between(recall_vals, precision_vals, alpha=0.1, color='green')
axes[1].set_title('Courbe Précision-Rappel', fontweight='bold')
axes[1].set_xlabel('Rappel (Sensibilité)')
axes[1].set_ylabel('Précision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{main_path}/tmp/courbes_roc_pr.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
print(f'AUC-ROC : {roc_auc:.4f}')
print(f'AP      : {ap:.4f}')

# **Grad-CAM + Bounding Box**

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer_name=None):
        """
        Args:
            model            : modèle Keras
            target_layer_name: nom de la couche conv cible (dernière conv par défaut)
        """
        self.model = model

        # Trouver automatiquement la dernière couche convolutionnelle
        if target_layer_name is None:
            target_layer_name = self._trouver_derniere_conv()
        self.target_layer_name = target_layer_name
        print(f'   Couche Grad-CAM : {target_layer_name}')

        # Modèle intermédiaire
        self.grad_model = Model(
            inputs  = model.inputs,
            outputs = [model.get_layer(target_layer_name).output,
                       model.output]
        )

    def _trouver_derniere_conv(self):
        """Trouve la dernière couche convolutionnelle du modèle"""
        for layer in reversed(self.model.layers):
            if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
                return layer.name
        raise ValueError('Aucune couche Conv2D trouvée dans le modèle')

    @tf.function
    def _compute_gradients(self, img_tensor):
        """Calcule les gradients par rapport à la couche cible"""
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_tensor)
            loss = predictions[:, 0]  # Sortie binaire
        gradients = tape.gradient(loss, conv_outputs)
        return conv_outputs, gradients, predictions

    def compute_heatmap(self, img_array, eps=1e-8):
        """
        Calcule la heatmap Grad-CAM pour une image.
        Args:
            img_array : array (1, H, W, 3) normalisé [0,1]
        Returns:
            heatmap   : array (H, W) en [0,1]
            pred_prob : probabilité prédite
        """
        img_tensor = tf.cast(img_array, tf.float32)
        conv_outputs, gradients, predictions = self._compute_gradients(img_tensor)

        # Pooling des gradients (importance par canal)
        pooled_grads = tf.reduce_mean(gradients, axis=(0, 1, 2))

        # Pondération des feature maps
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        # ReLU + normalisation
        heatmap = tf.maximum(heatmap, 0)
        heatmap = heatmap / (tf.reduce_max(heatmap) + eps)
        heatmap = heatmap.numpy()

        return heatmap, float(predictions[0, 0])

    def superposer(self, img_orig, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
        """
        Superpose la heatmap Grad-CAM sur l'image originale.
        Returns:
            superposed : image RGB uint8
            heatmap_rgb: heatmap colorisée
        """
        # Redimensionner heatmap à la taille de l'image
        h, w = img_orig.shape[:2]
        heatmap_resized = cv2.resize(heatmap, (w, h))

        # Appliquer colormap (JET : bleu→rouge)
        heatmap_uint8 = np.uint8(255 * heatmap_resized)
        heatmap_rgb   = cv2.applyColorMap(heatmap_uint8, colormap)
        heatmap_rgb   = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB)

        # Superposition
        if img_orig.max() <= 1.0:
            img_uint8 = np.uint8(img_orig * 255)
        else:
            img_uint8 = img_orig.copy()
        if len(img_uint8.shape) == 2:
            img_uint8 = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2RGB)

        superposed = cv2.addWeighted(img_uint8, 1 - alpha,
                                      heatmap_rgb, alpha, 0)
        return superposed, heatmap_rgb, heatmap_resized

    @staticmethod
    def bounding_box(heatmap_resized, threshold=0.5):
        """
        Calcule la bounding box autour de la zone d'activation.
        Args:
            threshold: seuil d'activation (0.5 par défaut)
        Returns:
            (x, y, w, h) de la boîte englobante
        """
        binary = (heatmap_resized > threshold).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            # Fallback : utiliser toute la région activée
            ys, xs = np.where(heatmap_resized > heatmap_resized.mean())
            if len(xs) > 0:
                return (xs.min(), ys.min(),
                        xs.max() - xs.min(), ys.max() - ys.min())
            return None
        # Union de tous les contours
        all_points = np.vstack(contours)
        return cv2.boundingRect(all_points)

In [ ]:
# Instancier GradCAM
grad_cam = GradCAM(model)

In [ ]:
def preparer_image(img_path, img_size=224):
    """Charge et prépare une image pour l'inférence"""
    img = load_img(img_path, target_size=(img_size, img_size), color_mode='rgb')
    arr = img_to_array(img) / 255.0
    return arr, np.array(img)

In [ ]:
def visualiser_gradcam(image_paths, labels, n_cols=3, bb_threshold=0.5):
    """
    Visualise Grad-CAM + bounding box pour une liste d'images.
    Affiche : image originale | heatmap | superposition + BB
    """
    n = len(image_paths)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    if n == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle('Grad-CAM — Visualisation des Régions d\'Intérêt',
                 fontsize=15, fontweight='bold', y=1.01)

    col_titles = ['Image Originale', 'Heatmap Grad-CAM', 'Superposition + Bounding Box']
    for ax, t in zip(axes[0], col_titles):
        ax.set_title(t, fontweight='bold', fontsize=12, pad=10)

    for row, (path, true_label) in enumerate(zip(image_paths, labels)):
        # Préparation
        arr_norm, arr_orig = preparer_image(path, IMG_SIZE)
        arr_batch = arr_norm[np.newaxis, ...]

        # Grad-CAM
        heatmap, pred_prob = grad_cam.compute_heatmap(arr_batch)
        superposed, heatmap_rgb, heatmap_resized = grad_cam.superposer(
            arr_norm, heatmap, alpha=0.45
        )

        # Bounding box
        bbox = grad_cam.bounding_box(heatmap_resized, threshold=bb_threshold)

        # Labels
        pred_label = 'PNEUMONIA' if pred_prob >= OPTIMAL_THRESH else 'NORMAL'
        correct    = '✅' if pred_label == true_label else '❌'
        color_pred = '#e74c3c' if pred_label == 'PNEUMONIA' else '#2ecc71'

        row_title = (f'{correct} Vrai: {true_label} | '
                     f'Prédit: {pred_label} ({pred_prob:.3f})')

        # ─ Col 1 : Originale ─
        axes[row][0].imshow(arr_orig)
        axes[row][0].axis('off')
        axes[row][0].set_ylabel(row_title, fontsize=10, color=color_pred,
                                 fontweight='bold', rotation=0,
                                 labelpad=180, va='center')

        # ─ Col 2 : Heatmap ─
        axes[row][1].imshow(heatmap_rgb)
        axes[row][1].axis('off')
        # Colorbar
        sm = plt.cm.ScalarMappable(cmap='jet',
                                    norm=plt.Normalize(vmin=0, vmax=1))
        plt.colorbar(sm, ax=axes[row][1], fraction=0.046, pad=0.04)

        # ─ Col 3 : Superposition + BB ─
        axes[row][2].imshow(superposed)
        if bbox is not None:
            x, y, w, h = bbox
            rect = mpatches.Rectangle(
                (x, y), w, h,
                linewidth=3, edgecolor='yellow',
                facecolor='none', linestyle='-'
            )
            axes[row][2].add_patch(rect)
            axes[row][2].text(
                x, y - 8, f'Zone d\'intérêt',
                color='yellow', fontsize=9,
                fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.5, pad=2)
            )
        axes[row][2].axis('off')

    plt.tight_layout()
    plt.savefig(f'{main_path}/tmp/gradcam_results.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Sélectionner des exemples représentatifs ──────────────────
print('🔍 Sélection des exemples pour Grad-CAM...')

test_normal_dir    = os.path.join(TEST_DIR, 'NORMAL')
test_pneumonia_dir = os.path.join(TEST_DIR, 'PNEUMONIA')

# 2 exemples NORMAL + 4 exemples PNEUMONIA
normal_imgs = [
    os.path.join(test_normal_dir, f)
    for f in random.sample(os.listdir(test_normal_dir), 2)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
][:2]

pneumonia_imgs = [
    os.path.join(test_pneumonia_dir, f)
    for f in random.sample(os.listdir(test_pneumonia_dir), 4)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
][:4]

# Fallback si os.listdir retourne des non-images
def filtrer_images(folder, n):
    imgs = [os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    return random.sample(imgs, min(n, len(imgs)))

normal_imgs    = filtrer_images(test_normal_dir, 2)
pneumonia_imgs = filtrer_images(test_pneumonia_dir, 4)

all_imgs   = normal_imgs + pneumonia_imgs
all_labels = ['NORMAL'] * len(normal_imgs) + ['PNEUMONIA'] * len(pneumonia_imgs)

print(f'   NORMAL    : {len(normal_imgs)} images')
print(f'   PNEUMONIA : {len(pneumonia_imgs)} images')

visualiser_gradcam(all_imgs, all_labels, bb_threshold=0.45)
print('✅ Grad-CAM sauvegardé')

# Sauvegarde du Modèle Final

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

In [ ]:
# ── 1. Modèle Keras (.keras) ──────────────────────────────────
model_path = os.path.join(SAVE_DIR, f'{MODEL_NAME}_{timestamp}.keras')
model.save(model_path)
print(f'✅ Modèle Keras sauvegardé : {model_path}')

In [ ]:
# ── 2. Poids uniquement (.weights.h5) ─────────────────────────
weights_path = os.path.join(SAVE_DIR, f'{MODEL_NAME}_{timestamp}.weights.h5')
model.save_weights(weights_path)
print(f'✅ Poids sauvegardés       : {weights_path}')

In [ ]:
# ── 3. Métadonnées JSON ───────────────────────────────────────
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_opt).ravel()
metadata = {
    'model_name'       : MODEL_NAME,
    'timestamp'        : timestamp,
    'architecture'     : 'DenseNet121',
    'img_size'         : IMG_SIZE,
    'batch_size'       : BATCH_SIZE,
    'epochs_phase1'    : len(history_p1.history['loss']),
    'epochs_phase2'    : len(history_p2.history['loss']),
    'optimal_threshold': float(OPTIMAL_THRESH),
    'class_indices'    : CLASS_INDICES,
    'test_metrics': {
        'accuracy'    : float(f'{test_acc:.4f}'),
        'auc_roc'     : float(f'{roc_auc:.4f}'),
        'f1_optimal'  : float(f'{OPTIMAL_F1:.4f}'),
        'sensitivity' : float(f'{tp/(tp+fn):.4f}'),
        'specificity' : float(f'{tn/(tn+fp):.4f}'),
        'precision'   : float(f'{tp/(tp+fp):.4f}'),
        'TP': int(tp), 'TN': int(tn),
        'FP': int(fp), 'FN': int(fn)
    },
    'train_samples': train_gen.n,
    'val_samples'  : val_gen.n,
    'test_samples' : test_gen.n
}

meta_path = os.path.join(SAVE_DIR, f'{MODEL_NAME}_{timestamp}_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f'✅ Métadonnées sauvegardées : {meta_path}')

In [ ]:
# ── Récapitulatif final ───────────────────────────────────────
print(f'   Accuracy     : {test_acc*100:6.2f}%')
print(f'   AUC-ROC      : {roc_auc:.4f}')
print(f'   F1 (optimal) : {OPTIMAL_F1:.4f}')
print(f'   Seuil optimal: {OPTIMAL_THRESH:.2f}')
print(f'   Sensibilité  : {tp/(tp+fn):.4f}  (vrais positifs PNEUMONIA)')
print(f'   Spécificité  : {tn/(tn+fp):.4f}  (vrais négatifs NORMAL)')
print('─' * 60)
print(f'   TP = {tp}  |  TN = {tn}  |  FP = {fp}  |  FN = {fn}')
print('=' * 60)

if test_acc >= 0.90:
    print('🎉 Objectif atteint : Accuracy > 90% !')
else:
    print(f'⚠️  Accuracy ({test_acc*100:.1f}%) < 90% — Fine-tuning supplémentaire recommandé')

# Test sur Image Externe

In [ ]:
def predire_image(img_path, model, threshold=0.5, afficher_gradcam=True, dpi=DPI):
    if not os.path.exists(img_path):
        print(f'❌ Image non trouvée : {img_path}')
        print('   → Uploadez votre image ou modifiez IMAGE_EXTERNE')
        return None

    # Préparation
    arr_norm, arr_orig = preparer_image(img_path, IMG_SIZE)
    arr_batch = arr_norm[np.newaxis, ...]

    # Prédiction
    pred_prob  = float(model.predict(arr_batch, verbose=0)[0, 0])
    pred_label = 'PNEUMONIA' if pred_prob >= threshold else 'NORMAL'
    confiance  = pred_prob if pred_label == 'PNEUMONIA' else (1 - pred_prob)

    # ── Grad-CAM ─────────────────────────────────────────────
    heatmap, _ = grad_cam.compute_heatmap(arr_batch)
    superposed, heatmap_rgb, heatmap_resized = grad_cam.superposer(
        arr_norm, heatmap, alpha=0.45
    )
    bbox = grad_cam.bounding_box(heatmap_resized, threshold=0.45)

    # ── Visualisation ────────────────────────────────────────
    color = '#e74c3c' if pred_label == 'PNEUMONIA' else '#2ecc71'
    icon  = '🔴' if pred_label == 'PNEUMONIA' else '✅'

    if afficher_gradcam:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(
            f'{icon}  Résultat : {pred_label}  |  '
            f'Confiance : {confiance*100:.1f}%  |  '
            f'Score pneumonie : {pred_prob:.4f}',
            fontsize=14, fontweight='bold', color=color
        )

        axes[0].imshow(arr_orig, cmap='gray' if len(arr_orig.shape)==2 else None)
        axes[0].set_title('Image Originale', fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(heatmap_rgb)
        axes[1].set_title('Heatmap Grad-CAM', fontweight='bold')
        axes[1].axis('off')
        sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(0,1))
        plt.colorbar(sm, ax=axes[1], fraction=0.046)

        axes[2].imshow(superposed)
        if bbox:
            x, y, w, h = bbox
            rect = mpatches.Rectangle((x, y), w, h,
                                       linewidth=3, edgecolor='yellow',
                                       facecolor='none')
            axes[2].add_patch(rect)
            axes[2].text(x, max(y-8, 0), 'Zone suspecte',
                         color='yellow', fontsize=10, fontweight='bold',
                         bbox=dict(facecolor='black', alpha=0.5))
        axes[2].set_title('Superposition + Zone d\'intérêt', fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(f'{main_path}/tmp/prediction_externe.png', dpi=dpi, bbox_inches='tight')
        plt.show()

    print('\n' + '═' * 50)
    print(f'  {icon}  DIAGNOSTIC : {pred_label}')
    print('─' * 50)
    print(f'  Score pneumonie : {pred_prob:.4f}')
    print(f'  Score normal    : {1-pred_prob:.4f}')
    print(f'  Confiance       : {confiance*100:.1f}%')
    print(f'  Seuil utilisé   : {threshold}')
    if pred_label == 'PNEUMONIA':
        niveau = 'élevée' if confiance > 0.8 else 'modérée' if confiance > 0.6 else 'faible'
        print(f'  Probabilité {niveau} de pneumonie')
        print('  ⚠️  Résultat à confirmer par un radiologue')
    else:
        print('  Radiographie sans signe évident de pneumonie')
        print('  ⚠️  Résultat à confirmer par un professionnel de santé')
    print('═' * 50)

    return {
        'label'       : pred_label,
        'probability' : pred_prob,
        'confidence'  : confiance
    }

In [ ]:
# ── Exécuter la prédiction ────────────────────────────────────
IMAGE_EXTERNE = f"{main_path}/test/Pneumonia_3.png"
result = predire_image(IMAGE_EXTERNE, model, threshold=OPTIMAL_THRESH, dpi=DPI)

In [ ]:
import shutil
from google.colab import files

# Créer une archive ZIP du dossier Models
output_filename = 'Models_archive'
shutil.make_archive(output_filename, 'zip', SAVE_DIR)

print(f'✅ Dossier "{SAVE_DIR}" zippé en "{output_filename}.zip"')

# Télécharger l'archive
files.download(f'{output_filename}.zip')
print('✅ Téléchargement lancé.')